In [1]:
import os
os.chdir('../..')
folder = 'trained_models/cls_embeddings'
os.getcwd()

'/home/trath/Code/nlpvise/src'

In [2]:
from skorch import NeuralNetClassifier

from training_pipeline import TrainingPipeline
from models.convlstm import ConvLSTM
from models.convnet import ConvNet
from models.bilstm import BiLSTM
from models.mlp import mlp

In [3]:
training_pipeline = TrainingPipeline()
training_pipeline.CLS = True
training_pipeline.load_data()
evaluation_results = {}

Data loaded successfully.
Train/Test split completed.


In [4]:
def evaluate_model(module, param_path):
    net = NeuralNetClassifier(module=module)
    net.initialize()
    net.load_params(f_params=param_path)
    net.module_.to(training_pipeline.device)

    return training_pipeline.eval_model(net, threshold=0.5)

In [5]:
with os.scandir(folder) as entries:
    for entry in entries:
        print(entry.name)
        match entry.name:
            case 'BiLSTM_CLS.pkl':
                results = evaluate_model(module=BiLSTM(input_size=768), param_path=entry.path)
                evaluation_results['BiLSTM'] = results
            case 'mlp_CLS.pkl':
                results = evaluate_model(module=mlp(input_size=768), param_path=entry.path)
                evaluation_results['MLP'] = results
            case 'ConvLSTM_CLS.pkl':
                results = evaluate_model(module=ConvLSTM(input_size=768), param_path=entry.path)
                evaluation_results['ConvLSTM'] = results
            case 'ConvNet_CLS.pkl':
                results = evaluate_model(module=ConvNet(input_size=768), param_path=entry.path)
                evaluation_results['ConvNet'] = results

ConvNet_CLS.pkl
BiLSTM_CLS.pkl
ConvLSTM_CLS.pkl
mlp_CLS.pkl


In [6]:
import pandas as pd
pd.DataFrame(evaluation_results).T

,ACC,MCC,F1,AUPRC,AUROC
ConvNet,0.781985,0.249408,0.714344,0.624560,0.742440
BiLSTM,0.779522,0.250712,0.716517,0.620743,0.739317
ConvLSTM,0.778546,0.237212,0.717522,0.614917,0.735462
MLP,0.773056,0.206250,0.700186,0.596626,0.713820
